# 3 What does this algorithm know?

You have used `ray.tune` twice — once on images in lesson 1, once on a composition family
in notebook 04 of this lesson. Both times it was a tool you pointed at a space. This
notebook opens it up.

There are only a few genuinely different ideas in hyperparameter search, and each one
exists because the previous one fails at something specific:

| | what it knows | where it fails |
|---|---|---|
| **grid** | nothing; you enumerate | the number of points explodes with the number of axes |
| **random** | nothing, but it samples the space instead of enumerating it | keeps sampling regions it has already learned are bad |
| **TPE** | a model of which regions scored well | still runs every trial to completion |
| **differential evolution** | a *population*, and how to recombine its members | no model of the surface; spends evaluations building one implicitly |
| **ASHA / Hyperband** | when to give up on a trial early | can discard a slow starter that would have won |

But there is a prior question that matters more than any of them, and notebook 02 already
gave us the tools to ask it: **is there anything in this space to find?**

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from notebooktester import param
from torch import nn
from vectormesh import VectorCache
from vectormesh.components import NeuralNet

BUDGET = param(24, test=4)
SEARCH_EPOCHS = param(5, test=1)
SEEDS = param((0, 1, 2), test=(0,))
BATCH, N_CLASSES = 128, 47
ENCODERS = ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"]

DATA = {}
for enc in ENCODERS:
    c = VectorCache.from_hub(f"pttrn-io/dtd-{enc}")
    d = c.dataset.shuffle(seed=0).with_format("numpy")   # shuffle before slicing -- see notebook 02
    X = torch.tensor(np.asarray(d["embed"], dtype="float32"))
    y = torch.tensor(np.asarray(d["label"]))
    a, b = int(0.60 * len(X)), int(0.80 * len(X))
    DATA[enc] = (X[:a], y[:a], X[a:b], y[a:b], X[b:], y[b:])

print(f"train/val/test = {len(DATA['dinov2-small'][0])}/{len(DATA['dinov2-small'][2])}/"
      f"{len(DATA['dinov2-small'][4])}   (a three-way split: the search never sees test)")

**Three-way split, and it matters here more than anywhere else in the course.** A search
tries dozens of configurations and keeps the best — which means the winner's score is the
maximum of dozens of noisy numbers, and the maximum of noisy numbers is biased upward.
Report that number as your result and you are reporting luck. The search optimises against
*validation*; the winner is scored once on *test*, at the end.

In [ ]:
WIDTHS, DEPTHS = [64, 128, 256, 512, 1024], [1, 2, 3]


def objective(width, depth, lr, encoder="dinov2-small", epochs=SEARCH_EPOCHS, seed=0, final=False):
    """Train a head, return accuracy. `final=True` scores on test instead of validation."""
    Xtr, ytr, Xva, yva, Xte, yte = DATA[encoder]
    torch.manual_seed(seed)
    model = NeuralNet([Xtr.shape[1]] + [int(width)] * int(depth), N_CLASSES)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr) - 1, BATCH):
            idx = perm[i:i + BATCH]
            opt.zero_grad()
            lossf(model(Xtr[idx]), ytr[idx]).backward()
            opt.step()
    model.eval()
    Xe, ye = (Xte, yte) if final else (Xva, yva)
    with torch.no_grad():
        return float((model(Xe).argmax(1) == ye).float().mean())

**Five epochs per trial, for every configuration.** Notebook 02 measured that this head
needs about 20 epochs to be done, so five is deliberately short — and that is fine *here*,
because search is a **ranking** problem, not a measurement problem. Every candidate gets
the same handicap, so the comparison between them stays fair.

That is the other half of the rule from notebook 02, and the distinction is worth being
precise about:

> While **searching**, an equal short budget is a legitimate economy. For any number you
> **report**, the budget must be justified. The winner of this search gets retrained
> properly before anyone quotes its score.

## 3.1 Grid: enumerate everything

The honest baseline. No cleverness: list the values, take the product, run them all.

In [ ]:
def grid_configs(with_encoder):
    encs = ENCODERS if with_encoder else ["dinov2-small"]
    return [dict(width=w, depth=d, lr=lr, encoder=e)
            for e in encs for w in WIDTHS for d in DEPTHS for lr in (1e-4, 1e-3, 1e-2)]


for we in (False, True):
    n = len(grid_configs(we))
    print(f"{'head + encoder' if we else 'head only':16s}: {n:4d} points to cover the grid, "
          f"budget is {BUDGET}  ->  you can afford {BUDGET / n:.0%}")

There it is, before a single model is trained. Adding **one** axis with three values
tripled the grid. This is the curse of dimensionality in its most concrete form: grid cost
is the *product* of the axes, so every axis you add multiplies the bill.

And when you cannot afford the whole grid, you truncate it — which is where grid does
something genuinely dangerous.

In [ ]:
def run_grid(budget, with_encoder, seed=0):
    cfgs = grid_configs(with_encoder)[:budget]      # truncated: the usual, silent, compromise
    hist = [(objective(**c), c) for c in cfgs]
    seen = sorted({c["encoder"] for c in cfgs})
    return hist, seen


hist, seen = run_grid(BUDGET, with_encoder=True)
best = max(hist, key=lambda r: r[0])
print(f"grid, encoder in the space: best val {best[0]:.4f} with {best[1]['encoder']}")
print(f"encoders actually tried: {seen}")

**It never tried two of the three encoders.** The truncated grid spent its entire budget
inside the first value of the first axis, and nothing warned you. The axis that notebook
02 measured as worth ~15 points was in the search space, and the search never looked at it.

Nothing about this is exotic — it is what "just run the first N of the grid" does, and the
ordering of your nested loops silently decides what gets explored. Grid's one virtue is
that it is perfectly reproducible; here that just means it reliably makes the same mistake.

## 3.2 Random: sample instead of enumerate

Bergstra and Bengio (2012) made the argument that displaced grid search almost everywhere:
if only some axes matter, a grid wastes its budget re-testing the irrelevant ones at every
level of the relevant ones, while random search gives you a distinct value of *every* axis
on *every* trial.

In [ ]:
def sample(rng, with_encoder):
    return dict(width=int(rng.choice(WIDTHS)),
                depth=int(rng.choice(DEPTHS)),
                lr=float(10 ** rng.uniform(-4, -2)),
                encoder=str(rng.choice(ENCODERS)) if with_encoder else "dinov2-small")


def run_random(budget, with_encoder, seed=0):
    rng = np.random.default_rng(seed)
    return [(lambda c: (objective(**c), c))(sample(rng, with_encoder)) for _ in range(budget)]

## 3.3 Differential evolution: keep a population

The family that has no model of the response surface at all, and instead keeps a set of
candidates that breed. A new candidate is built by taking one member and adding the
*difference* between two others — so the population's own spread sets the step size. Early
on, when members are scattered, steps are large; as they converge, steps shrink
automatically.

Twenty lines, no library, and it is the ancestor of most swarm methods:

In [ ]:
def run_de(budget, with_encoder, seed=0, pop=8, F=0.7, CR=0.9):
    rng = np.random.default_rng(seed)
    hi3 = (len(ENCODERS) - 1) if with_encoder else 0
    lo = np.array([0, 0, -4, 0], float)
    hi = np.array([len(WIDTHS) - 1, len(DEPTHS) - 1, -2, hi3], float)

    def decode(v):
        v = np.clip(v, lo, hi)
        return dict(width=WIDTHS[int(round(v[0]))], depth=DEPTHS[int(round(v[1]))],
                    lr=float(10 ** v[2]),
                    encoder=ENCODERS[int(round(v[3]))] if with_encoder else "dinov2-small")

    P = rng.uniform(lo, hi, size=(pop, 4))
    scores = [objective(**decode(p)) for p in P]
    hist = [(s, decode(p)) for s, p in zip(scores, P)]
    while len(hist) < budget:
        for i in range(pop):
            if len(hist) >= budget:
                break
            a, b, c = rng.choice([j for j in range(pop) if j != i], 3, replace=False)
            trial = np.where(rng.random(4) < CR, P[a] + F * (P[b] - P[c]), P[i])
            s = objective(**decode(trial))
            hist.append((s, decode(trial)))
            if s > scores[i]:
                P[i], scores[i] = trial, s
    return hist

A population of 8 inside a budget of 24 leaves only two generations, which is barely
enough for the recombination to do anything. That is a real limitation of population
methods and worth naming: **they need evaluations to earn their keep.** On a cache, where
a trial costs a couple of seconds, that is affordable — a population of 100 is a couple of
minutes. On a task where a trial costs an hour, it is not.

## 3.4 Round one: search the head, with the encoder fixed

In [ ]:
def compare(with_encoder, budget=BUDGET):
    out = {}
    for name, fn in [("grid", lambda s: run_grid(budget, with_encoder, s)[0]),
                     ("random", lambda s: run_random(budget, with_encoder, s)),
                     ("DE", lambda s: run_de(budget, with_encoder, s))]:
        bests, curves, picks = [], [], []
        for s in SEEDS:
            h = fn(s)
            bests.append(max(r[0] for r in h))
            curves.append(np.maximum.accumulate([r[0] for r in h]))
            picks.append(max(h, key=lambda r: r[0])[1])
        out[name] = (float(np.mean(bests)), float(np.std(bests)), np.mean(curves, 0), picks)
    return out


t0 = time.time()
round1 = compare(with_encoder=False)
for name, (m, s, _, _) in round1.items():
    print(f"{name:8s} best val {m:.4f} +- {s:.4f}")
print(f"({time.time() - t0:.0f}s)")

**They tie.** The spread between the three algorithms is about the same size as the spread
between seeds of a single algorithm — which means that on this space, the choice of search
algorithm carries no information.

That is not a disappointing result, it is the one notebook 02 predicted. The head-capacity
axis was measured there as worth about two points in total, non-monotone, with width
buying a little and depth taking it back. **A search cannot find structure that is not
there**, and no amount of algorithmic sophistication changes that.

Notice also that grid's standard deviation is exactly zero. It is deterministic — it does
the same thing every time — which reads as stability and is really just blindness.

## 3.5 Round two: put the encoder in the space

Notebook 02 measured the encoder axis as worth ~15 points, an order of magnitude more than
the head. So the obvious move is to let the search choose it. Predict the outcome before
you run this: **does adding a high-impact axis make the search better?**

In [ ]:
t0 = time.time()
round2 = compare(with_encoder=True)
print(f"{'':8s} {'round 1 (head)':>18s} {'round 2 (+encoder)':>20s}")
for name in round1:
    m1, s1, _, _ = round1[name]
    m2, s2, _, picks = round2[name]
    print(f"{name:8s} {m1:12.4f} +-{s1:.4f} {m2:14.4f} +-{s2:.4f}   picked: "
          f"{sorted({p['encoder'] for p in picks})}")
print(f"({time.time() - t0:.0f}s)")

Two things happened, and the second one is counterintuitive.

**The search found the right encoder.** Every algorithm's winning configuration uses
`dinov2-small` — the one notebook 02 measured as best by fifteen points. The search works:
handed an axis that matters, it lands on the right value without being told.

**And the best score got slightly *worse*.** Not better. Adding a fifteen-point axis to the
search space made the outcome marginally worse than leaving it out.

The reason is budget. Two of the three encoders are known to be far worse, so roughly
two-thirds of every sample is spent evaluating configurations that cannot win. With 24
evaluations, that is 16 wasted trials — and the head axis they *would* have explored was
worth little anyway, so nothing compensates.

> **A search space is a claim about where the answer might be.** Putting a region in it
> that you already know is bad does not make the search more thorough; it makes it poorer,
> because budget is conserved.

This is the practical lesson of the notebook, and it is the opposite of the instinct that
says a wider search is a safer search. The search space is a modelling decision, and
notebook 02 — measuring which axes move the metric before searching over them — is how you
make it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (res, title) in zip(axes, [(round1, "round 1: head only"),
                                   (round2, "round 2: head + encoder")]):
    for name, (_, _, curve, _) in res.items():
        ax.plot(range(1, len(curve) + 1), curve, marker="o", ms=3, label=name)
    ax.set_xlabel("evaluations"); ax.set_title(title); ax.grid(alpha=.3)
axes[0].set_ylabel("best validation accuracy so far"); axes[0].legend()
plt.suptitle("best-so-far: the shape to read, not the final number")
plt.tight_layout(); plt.show()

Read the *shape*, not the endpoint. A best-so-far curve that is still climbing means the
budget was the binding constraint and more evaluations would help. A curve that flattened
early means the space was exhausted, and more budget would buy nothing — you need a
different space, not a longer search.

## 3.6 The number you are allowed to report

The winner of a search has the highest of two dozen noisy validation scores, so its
validation number is optimistically biased almost by construction. Retrain it properly —
at the epoch count notebook 02 justified — and score it once, on the test split the search
never touched.

In [ ]:
_, _, _, picks = round2["random"]
winner = picks[0]
print("winner:", winner)

val_reported = objective(**winner)
test_honest = objective(**winner, epochs=param(20, test=1), final=True)
print(f"\nvalidation, as the search saw it (5 epochs) : {val_reported:.4f}")
print(f"test, retrained properly (20 epochs)        : {test_honest:.4f}")

Two changes at once — a different split *and* a longer schedule — so whatever gap you see
is not a clean measurement of selection bias. If the two numbers come out close, or even
identical, that is luck rather than proof that the bias is absent: with two dozen trials
the optimistic bias is usually small, but "usually small" is not a guarantee you can
report.

**The point is the discipline, not the delta.** The number you publish comes from data the
search never saw, at a budget you justified. You do not get to check whether that mattered
*after* deciding to skip it.

One detail worth noticing in the winning configuration: the search chose a **wide, shallow**
head — large width, depth 1. That is exactly what notebook 02 measured independently, where
width bought about a point and a half and depth gave it back. Two different methods, same
answer, which is the kind of agreement that should raise your confidence in both.

## What to take from this

- **Grid's cost is the product of its axes**, and a truncated grid explores silently and
  badly — here it never tried two of three encoders.
- **Random beats grid for the reason Bergstra and Bengio gave**: every trial gives a fresh
  value of every axis, so irrelevant axes cost you nothing.
- **A model of the surface (TPE) or a population (DE) buys you something only when there
  is a surface worth modelling.** On a flat axis all three algorithms tie.
- **The space matters more than the algorithm.** Adding a known-bad region made the result
  worse, because budget is conserved.
- **Report the test number, from a properly trained winner.**

Notebook 04 takes the next step: what if you spend the budget unevenly *within* a trial —
stopping the hopeless ones early — and what does that buy on a Pareto front where accuracy
is not the only axis?

## Exercises

**E1 — reorder the grid.** `grid_configs` loops encoder-outermost. Move it innermost and
re-run §3.1. The budget is unchanged and the grid is the same size; does the truncated grid
now find a better configuration? What does that tell you about how much of grid's result is
a property of your loop nesting rather than of your data?

**E2 — give the population room.** Re-run DE with a budget of 100 and a population of 20.
Does it pull ahead of random once it has generations to work with? Keep random at the same
budget — equal evaluations is the control.

**E3 — a space worth searching.** Both rounds searched axes notebook 02 showed to be
nearly flat. Build a space where something *is* at stake: train on 10% of the rows (where
§2.3 showed the data axis is steep) and search over regularisation — weight decay,
`GaussianNoise` sigma, dropout. Do the algorithms separate now?

**E4 — the honest-number gap.** §3.6 changed two things at once. Separate them: score the
winner on test at 5 epochs, and on validation at 20. Which of the two changes explains
more of the difference?

**E5 — how much budget did the encoder axis cost?** Round 2 sampled three encoders when
one would do. Estimate the loss directly: run random search with the encoder axis, but
count only the trials that happened to draw `dinov2-small`. How does that curve compare
with round 1's at the same *number of useful evaluations*?